In [5]:
"""
Cartea-Jaimungal-Ricci (CJR) Hawkes Optimal Execution Model
============================================================
Paper : Cartea, Jaimungal & Ricci (2018)
        "Algorithmic Trading, Stochastic Control, and Mutually Exciting Processes"
        SIAM Review, Vol. 60, No. 3, pp. 673-703

Desk problem solved:
  Execution and market-making desks model order arrivals as independent
  Poisson processes. In reality trades SELF-EXCITE (a burst of buys triggers
  more buys) and CROSS-EXCITE (buys eventually trigger defensive sells).
  This order-flow clustering is called the Hawkes effect and it means:
    1. Fill rates are grossly underestimated during clustering events
       (a naive desk posts too deep and misses profitable fills)
    2. The imbalance between buy and sell intensities carries a measurable
       short-term alpha signal for mid-price direction

  CJR (2018) models this with a bivariate Hawkes process, derives the
  HJB-optimal posting depth analytically, and shows that HFT traders who
  ignore short-term alpha are driven out by better-informed competitors.

PART 1  CJR replication: bivariate Hawkes intensities, alpha signal, and
        Hawkes-aware posting depth that tightens during clustering events

PART 2  Three original extensions that address assumptions the paper makes:
  (a) Asymmetric decay kernel: CJR assumes buy and sell excitation decay at
      the same rate (beta). Empirically, buy-side clustering persists longer
      than sell-side (documented in equity LOBs). We allow separate beta_buy
      and beta_sell, improving fill-rate estimates on each side independently.
  (b) HMM regime-switching baseline: CJR uses a constant baseline intensity mu.
      We replace it with a 2-state hidden Markov model (calm / active) detected
      online via exponential smoothing of the observed intensity. The posting
      depth adapts immediately to regime transitions.
  (c) Cross-asset intensity spillover: a correlated asset's order flow feeds
      into the primary instrument's Hawkes baseline, giving early-warning of
      imminent clustering before it appears in the primary instrument's own flow.
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────
#  GLOBAL PARAMETERS
# ─────────────────────────────────────────────────
T        = 1.0
DT       = 0.005
N_STEPS  = int(T / DT)       # 200 steps per session

MU_BASE  = 8.0               # baseline order arrival intensity
ALPHA_S  = 0.45              # self-excitation (same-side clustering)
ALPHA_C  = 0.20              # cross-excitation (opposite-side response)
BETA     = 4.0               # excitation decay rate (symmetric, Part 1)

GAMMA    = 0.05              # risk aversion
SIGMA    = 0.008             # mid-price vol per step
A        = 120.0             # fill intensity scale
K        = 1.2               # fill intensity decay with depth
Q_MAX    = 8                 # inventory limit (lots)
S0       = 100.0             # initial mid-price

# Extension parameters
CALM_MU  = 5.0
ACTIVE_MU= 18.0
SMOOTH   = 0.12              # HMM smoothing coefficient
CROSS_W  = 0.18              # cross-asset spillover weight


# ============================================================
#  PART 1 -- CJR BIVARIATE HAWKES MODEL
# ============================================================

class BivarHawkes:
    """
    Bivariate (buy/sell) Hawkes process.

    Conditional intensities at time t:
        lambda_buy(t)  = mu + alpha_s * R_buy(t)  + alpha_c * R_sell(t)
        lambda_sell(t) = mu + alpha_s * R_sell(t) + alpha_c * R_buy(t)

    Cluster term recursion (exact for piecewise-constant events in dt):
        R_buy(t+dt)  = [ R_buy(t)  + events_buy(t)  ] * exp(-beta * dt)
        R_sell(t+dt) = [ R_sell(t) + events_sell(t) ] * exp(-beta * dt)

    Stability condition: alpha_s + alpha_c < beta  (branching ratio < 1)
    """

    def __init__(self, mu=MU_BASE, alpha_s=ALPHA_S, alpha_c=ALPHA_C, beta=BETA,
                 beta_buy=None, beta_sell=None,
                 alpha_s_buy=None, alpha_s_sell=None):
        self.mu       = mu
        self.alpha_s  = alpha_s
        self.alpha_c  = alpha_c
        self.beta     = beta
        # Extension 2a: asymmetric decay
        self.beta_buy      = beta_buy      or beta
        self.beta_sell     = beta_sell     or beta
        self.alpha_s_buy   = alpha_s_buy   or alpha_s
        self.alpha_s_sell  = alpha_s_sell  or alpha_s

    def intensities(self, R_buy, R_sell, mu_eff=None, asym=False):
        mu = mu_eff if mu_eff is not None else self.mu
        if asym:
            lb = mu + self.alpha_s_buy  * R_buy  + self.alpha_c * R_sell
            ls = mu + self.alpha_s_sell * R_sell + self.alpha_c * R_buy
        else:
            lb = mu + self.alpha_s * R_buy  + self.alpha_c * R_sell
            ls = mu + self.alpha_s * R_sell + self.alpha_c * R_buy
        return max(lb, 0.01), max(ls, 0.01)

    def update(self, R_buy, R_sell, ev_buy, ev_sell, asym=False):
        if asym:
            R_b = (R_buy  + ev_buy ) * np.exp(-self.beta_buy  * DT)
            R_s = (R_sell + ev_sell) * np.exp(-self.beta_sell * DT)
        else:
            d   = np.exp(-self.beta * DT)
            R_b = (R_buy  + ev_buy ) * d
            R_s = (R_sell + ev_sell) * d
        return R_b, R_s


class CJRExecutor:
    """
    CJR optimal posting depth using real-time Hawkes intensity.

    Core insight (CJR Prop. 3): the optimal posting depth is:
        delta*(t, q, lambda) = base_depth
                             - depth_tightening(lambda)   [Hawkes correction]
                             +/- alpha_skew(q, alpha_t)   [directional skew]

    where:
        base_depth = (1/k)*ln(1 + k/gamma) + gamma*sigma^2*(T-t)/2
        depth_tightening = c * log( (lb+ls)/(2*mu) )  [tighten when busy]
        alpha_skew = q*gamma*sigma^2*(T-t)*alpha*0.4   [skew for direction]

    The key innovation vs naive Poisson: during a clustering event
    (lb >> mu), the Hawkes-aware executor posts shallower, capturing fills
    that the naive executor misses by posting at the stale baseline depth.
    """

    def __init__(self, gamma=GAMMA, sigma=SIGMA, k=K, A=A, T=T, q_max=Q_MAX):
        self.gamma = gamma; self.sigma = sigma
        self.k = k; self.A = A; self.T = T; self.q_max = q_max

    def base_depth(self, t):
        return ((1.0 / self.k) * np.log(1.0 + self.k / self.gamma)
                + self.gamma * self.sigma**2 * (self.T - t) / 2.0)

    def alpha_signal(self, lb, ls):
        return (lb - ls) / (lb + ls + 1e-10)

    def posting_depth(self, t, q, lb, ls, mu_eff, mode='poisson'):
        bd = self.base_depth(t)
        if mode == 'poisson':
            return bd, bd
        # Hawkes depth correction: tighten when intensity is elevated
        ratio     = (lb + ls) / (2.0 * mu_eff + 1e-10)
        hawk_adj  = -0.10 * np.log(max(ratio, 0.1))
        if mode == 'hawkes':
            db = da = max(bd + hawk_adj, 0.002)
        else:   # hawkes+alpha+hmm+cross
            alpha = self.alpha_signal(lb, ls)
            q_c   = np.clip(q, -self.q_max, self.q_max)
            skew  = q_c * self.gamma * self.sigma**2 * (self.T - t) * alpha * 0.4
            db    = max(bd + hawk_adj - skew, 0.002)
            da    = max(bd + hawk_adj + skew, 0.002)
        return db, da

    def simulate(self, hawkes, n_paths=1, seed=0, mode='poisson',
                 asym=False, hmm=False, cross_asset=False, directional=True):
        rng = np.random.default_rng(seed)
        results = []

        for p in range(n_paths):
            s=S0; q=0; cash=0
            R_b=0; R_s=0; R_cross=0
            smooth_lam = hawkes.mu

            s_p, q_p, pnl_p = [s], [q], [0.0]
            lb_p, ls_p, alpha_p, db_p, da_p, reg_p = [], [], [], [], [], []

            for i in range(N_STEPS):
                t   = i * DT
                q_c = float(np.clip(q, -self.q_max, self.q_max))

                # HMM regime detection (extension 2b)
                if hmm:
                    smooth_lam = (1-SMOOTH)*smooth_lam + SMOOTH*(
                        hawkes.mu + hawkes.alpha_s*R_b)
                    regime = 1 if smooth_lam > (CALM_MU + ACTIVE_MU)/2 else 0
                    mu_eff = ACTIVE_MU if regime == 1 else CALM_MU
                else:
                    mu_eff = hawkes.mu; regime = 0

                # Cross-asset spillover (extension 2c)
                if cross_asset:
                    cross_ev = rng.poisson((CALM_MU + 0.3*R_cross)*DT)
                    R_cross  = (R_cross + cross_ev)*np.exp(-BETA*DT)
                    mu_eff  += CROSS_W * R_cross

                lb, ls = hawkes.intensities(R_b, R_s, mu_eff=mu_eff, asym=asym)

                # Posting depths
                db, da = self.posting_depth(t, q_c, lb, ls, mu_eff, mode=mode)

                # Fill events (Poisson with depth-adjusted rate)
                n_buy  = min(rng.poisson(self.A * np.exp(-self.k*db) * DT),
                             max(0, self.q_max - int(q_c)))
                n_sell = min(rng.poisson(self.A * np.exp(-self.k*da) * DT),
                             max(0, self.q_max + int(q_c)))

                # Background LOB order flow (drives Hawkes; directional run injects buy surge)
                if directional and 0.28 < t < 0.58:
                    ev_b = rng.poisson(lb * 2.2 * DT)
                    ev_s = rng.poisson(ls * DT)
                else:
                    ev_b = rng.poisson(lb * DT)
                    ev_s = rng.poisson(ls * DT)

                R_b, R_s = hawkes.update(R_b, R_s, ev_b, ev_s, asym=asym)

                q    += n_buy - n_sell
                cash += n_sell*(s + da) - n_buy*(s - db)
                s    += rng.standard_normal() * self.sigma

                s_p.append(s); q_p.append(q)
                pnl_p.append(cash + q*s)
                lb_p.append(lb); ls_p.append(ls)
                alpha_p.append(self.alpha_signal(lb, ls))
                db_p.append(db); da_p.append(da)
                reg_p.append(regime)

            cash += q * s    # flat at session end
            pnl_p[-1] = cash

            results.append(dict(
                s=np.array(s_p), q=np.array(q_p), pnl=np.array(pnl_p),
                lb=np.array(lb_p), ls=np.array(ls_p),
                alpha=np.array(alpha_p),
                depth_b=np.array(db_p), depth_a=np.array(da_p),
                regime=np.array(reg_p),
                final_pnl=cash, final_q=q_p[-1]
            ))
        return results


# ============================================================
#  VISUALISATION
# ============================================================

C = dict(blue="#185FA5", teal="#0F6E56", coral="#D85A30", amber="#BA7517",
         gray="#5F5E5A", lgray="#D3D1C7", purple="#534AB7", red="#A32D2D")

def plot_all():
    hwk      = BivarHawkes()
    hwk_asym = BivarHawkes(alpha_s_buy=0.58, alpha_s_sell=0.28,
                            beta_buy=2.8, beta_sell=5.8)
    ex = CJRExecutor()

    r_poi  = ex.simulate(hwk,      seed=7, mode='poisson',     directional=True)[0]
    r_cjr  = ex.simulate(hwk,      seed=7, mode='hawkes',      directional=True)[0]
    r_hmm  = ex.simulate(hwk,      seed=7, mode='hawkes',  hmm=True, directional=True)[0]
    r_full = ex.simulate(hwk_asym, seed=7, mode='hawkes+alpha+ext', asym=True,
                          hmm=True, cross_asset=True, directional=True)[0]

    mc_poi  = ex.simulate(hwk,      n_paths=500, seed=0, mode='poisson',  directional=True)
    mc_cjr  = ex.simulate(hwk,      n_paths=500, seed=0, mode='hawkes',   directional=True)
    mc_full = ex.simulate(hwk_asym, n_paths=500, seed=0, mode='hawkes+alpha+ext',
                           asym=True, hmm=True, cross_asset=True, directional=True)

    t_step = np.linspace(0, T, N_STEPS + 1)
    t_spr  = np.linspace(0, T, N_STEPS)

    fig = plt.figure(figsize=(20, 15), facecolor="black")
    
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.47, wspace=0.36)

    def sty(ax, title, xl, yl):
        ax.set_title(title, fontsize=11, fontweight="bold", pad=5)
        ax.set_xlabel(xl, fontsize=9); ax.set_ylabel(yl, fontsize=9)
        ax.grid(True, alpha=0.22, lw=0.7); ax.tick_params(labelsize=8)

    # 1. Hawkes intensities with buy surge
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(t_spr, r_cjr["lb"], color=C["blue"],  lw=1.8, label="lambda buy")
    ax1.plot(t_spr, r_cjr["ls"], color=C["coral"], lw=1.8, label="lambda sell")
    ax1.axhline(MU_BASE, color=C["gray"], lw=0.9, ls="--", alpha=0.7, label="baseline mu")
    ax1.axvspan(0.28, 0.58, alpha=0.09, color=C["blue"], label="directional buy run")
    sty(ax1, "Hawkes conditional intensities (CJR)", "time", "lambda (arrivals / dt)")
    ax1.legend(fontsize=7.5, framealpha=0)

    # 2. Alpha signal (short-term directional forecast)
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.fill_between(t_spr, r_cjr["alpha"], 0,
                     where=(r_cjr["alpha"] > 0), alpha=0.5, color=C["blue"])
    ax2.fill_between(t_spr, r_cjr["alpha"], 0,
                     where=(r_cjr["alpha"] < 0), alpha=0.5, color=C["coral"])
    ax2.axhline(0, color=C["gray"], lw=0.7, ls="--")
    ax2.axvspan(0.28, 0.58, alpha=0.08, color=C["blue"])
    sty(ax2, "Short-term alpha: (lb-ls)/(lb+ls)", "time", "alpha [-1, +1]")
    ax2.text(0.30, 0.88, "buy pressure = upward drift", transform=ax2.transAxes,
             fontsize=8, color=C["blue"])

    # 3. Asymmetric intensities (extension 2a)
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.plot(t_spr, r_cjr["lb"],  color=C["blue"],  lw=1.5, ls="--", alpha=0.6,
             label="buy (sym.)")
    ax3.plot(t_spr, r_full["lb"], color=C["blue"],  lw=2.0,
             label="buy (asym. ext.)")
    ax3.plot(t_spr, r_cjr["ls"],  color=C["coral"], lw=1.5, ls="--", alpha=0.6,
             label="sell (sym.)")
    ax3.plot(t_spr, r_full["ls"], color=C["coral"], lw=2.0,
             label="sell (asym. ext.)")
    ax3.axvspan(0.28, 0.58, alpha=0.08, color=C["blue"])
    sty(ax3, "Asymmetric decay: buys cluster longer (ext. 2a)", "time", "lambda")
    ax3.legend(fontsize=7, framealpha=0, ncol=2)

    # 4. Posting depths: naive vs Hawkes vs full extension
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.plot(t_spr, r_poi["depth_b"],  color=C["gray"],   lw=1.5, ls="--",
             label="naive Poisson depth")
    ax4.plot(t_spr, r_cjr["depth_b"],  color=C["blue"],   lw=2.0,
             label="CJR bid (Hawkes-aware)")
    ax4.plot(t_spr, r_cjr["depth_a"],  color=C["coral"],  lw=2.0,
             label="CJR ask (Hawkes-aware)")
    ax4.plot(t_spr, r_full["depth_b"], color=C["teal"],   lw=1.5, ls="-.",
             label="full ext. bid")
    ax4.axvspan(0.28, 0.58, alpha=0.07, color=C["blue"])
    sty(ax4, "Posting depth: Poisson vs Hawkes-aware", "time", "depth from mid ($)")
    ax4.legend(fontsize=7.5, framealpha=0)

    # 5. HMM regime detection (extension 2b)
    ax5 = fig.add_subplot(gs[1, 1])
    regime_scaled = np.array(r_hmm["regime"]) * ACTIVE_MU
    ax5.fill_between(t_spr, 0, regime_scaled, alpha=0.22, color=C["amber"],
                     label="detected: active")
    ax5.plot(t_spr, r_hmm["lb"], color=C["blue"], lw=1.8, label="lambda_buy (HMM)")
    ax5.plot(t_spr, r_cjr["lb"], color=C["gray"], lw=1.0, ls="--", alpha=0.6,
             label="lambda_buy (static)")
    ax5.axhline(CALM_MU,   color=C["teal"], lw=0.8, ls=":", alpha=0.8)
    ax5.axhline(ACTIVE_MU, color=C["red"],  lw=0.8, ls=":", alpha=0.8)
    sty(ax5, "HMM regime detection (ext. 2b)", "time", "intensity")
    ax5.legend(fontsize=7.5, framealpha=0)

    # 6. Inventory comparison
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.plot(t_step, r_poi["q"],  color=C["gray"],  lw=1.8, ls="--", label="naive Poisson")
    ax6.plot(t_step, r_cjr["q"],  color=C["blue"],  lw=2.0, label="CJR Hawkes")
    ax6.plot(t_step, r_full["q"], color=C["teal"],  lw=2.0, label="full extension")
    ax6.axhline(0, color=C["gray"], lw=0.7, ls="--")
    ax6.axvspan(0.28, 0.58, alpha=0.07, color=C["blue"])
    ax6.axhspan( Q_MAX-0.5,  Q_MAX+0.5, alpha=0.1, color=C["red"])
    ax6.axhspan(-Q_MAX-0.5, -Q_MAX+0.5, alpha=0.1, color=C["red"])
    sty(ax6, "Inventory during directional run", "time", "inventory (lots)")
    ax6.legend(fontsize=7.5, framealpha=0)

    # 7. P&L distribution (MC)
    bp = np.array([r["final_pnl"] for r in mc_poi])
    cp = np.array([r["final_pnl"] for r in mc_cjr])
    fp = np.array([r["final_pnl"] for r in mc_full])
    ax7 = fig.add_subplot(gs[2, 0])
    lo = min(bp.min(), cp.min(), fp.min()) - 1
    hi = max(bp.max(), cp.max(), fp.max()) + 1
    bins = np.linspace(lo, hi, 46)
    ax7.hist(bp, bins=bins, color=C["gray"],  alpha=0.55,
             label=f"Poisson  mean={bp.mean():.2f}")
    ax7.hist(cp, bins=bins, color=C["blue"],  alpha=0.55,
             label=f"CJR       mean={cp.mean():.2f}")
    ax7.hist(fp, bins=bins, color=C["teal"],  alpha=0.55,
             label=f"full ext. mean={fp.mean():.2f}")
    sty(ax7, "P&L distribution (MC, 500 paths)", "final P&L ($)", "count")
    ax7.legend(fontsize=7.5, framealpha=0)

    # 8. Final inventory distribution
    bq = np.array([r["final_q"] for r in mc_poi])
    cq = np.array([r["final_q"] for r in mc_cjr])
    fq = np.array([r["final_q"] for r in mc_full])
    ax8 = fig.add_subplot(gs[2, 1])
    lim = int(max(abs(bq).max(), abs(cq).max(), abs(fq).max())) + 2
    binsq = np.arange(-lim-0.5, lim+1.5, 1)
    ax8.hist(bq, bins=binsq, color=C["gray"],  alpha=0.55, rwidth=0.5,
             label=f"Poisson  std={bq.std():.2f}")
    ax8.hist(cq, bins=binsq, color=C["blue"],  alpha=0.55, rwidth=0.5,
             label=f"CJR       std={cq.std():.2f}")
    ax8.hist(fq, bins=binsq, color=C["teal"],  alpha=0.55, rwidth=0.5,
             label=f"full ext. std={fq.std():.2f}")
    ax8.axvline(0, color=C["gray"], lw=1.0, ls="--")
    sty(ax8, "End-of-session inventory (MC)", "lots", "count")
    ax8.legend(fontsize=7.5, framealpha=0)

    # 9. Hawkes branching matrix
    ax9 = fig.add_subplot(gs[2, 2])
    mat = np.array([[ALPHA_S/BETA, ALPHA_C/BETA],
                    [ALPHA_C/BETA, ALPHA_S/BETA]])
    im = ax9.imshow(mat, cmap="Blues", aspect="auto", vmin=0, vmax=0.15)
    ax9.set_xticks([0,1]); ax9.set_yticks([0,1])
    ax9.set_xticklabels(["buy event", "sell event"], fontsize=9)
    ax9.set_yticklabels(["buy intensity", "sell intensity"], fontsize=9)
    for i in range(2):
        for j in range(2):
            ax9.text(j, i, f"{mat[i,j]:.3f}", ha="center", va="center",
                     fontsize=12, fontweight="500",
                     color="white" if mat[i,j] > 0.08 else "#2C2C2A")
    plt.colorbar(im, ax=ax9, fraction=0.046, pad=0.04)
    ax9.set_title("Hawkes excitation matrix (branching ratios)",
                  fontsize=11, fontweight="bold", pad=5)
    ax9.text(0.5, -0.14, f"sum = {(ALPHA_S+ALPHA_C)/BETA:.2f} < 1 (stationary)",
             ha="center", transform=ax9.transAxes, fontsize=8, color=C["gray"])

    

    out = "cartea_jaimungal_ricci_hawkes.png"
    plt.savefig(out, dpi=155, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"Saved: {out}")


def print_summary():
    hwk      = BivarHawkes()
    hwk_asym = BivarHawkes(alpha_s_buy=0.58, alpha_s_sell=0.28,
                            beta_buy=2.8, beta_sell=5.8)
    ex = CJRExecutor()

    mc_poi  = ex.simulate(hwk,      n_paths=600, seed=5, mode='poisson',  directional=True)
    mc_cjr  = ex.simulate(hwk,      n_paths=600, seed=5, mode='hawkes',   directional=True)
    mc_full = ex.simulate(hwk_asym, n_paths=600, seed=5, mode='hawkes+alpha+ext',
                           asym=True, hmm=True, cross_asset=True, directional=True)

    def stats(mc):
        p  = np.array([r["final_pnl"] for r in mc])
        q  = np.array([r["final_q"]   for r in mc])
        sh = p.mean() / (p.std() + 1e-9)
        return p.mean(), p.std(), sh, q.std(), np.percentile(p, 5)

    b, c, f = stats(mc_poi), stats(mc_cjr), stats(mc_full)

    print("=" * 68)
    print("  CJR HAWKES EXECUTION  (directional-run scenario, 600 paths)")
    print("=" * 68)
    print(f"  {'Model':<24} {'Mean PnL':>9} {'Std':>7} {'Sharpe':>8} {'InvStd':>8} {'VaR5%':>9}")
    print("-" * 68)
    for nm, s in [("Naive Poisson", b), ("CJR Hawkes (Part 1)", c), ("Full extension (Part 2)", f)]:
        print(f"  {nm:<24} {s[0]:>9.3f} {s[1]:>7.3f} {s[2]:>8.3f} "
              f"{s[3]:>8.3f} {s[4]:>9.3f}")
    print("=" * 68)
    print(f"  CJR vs Poisson   Sharpe gain: {(c[2]-b[2])/abs(b[2])*100:+.1f}%")
    print(f"  Full ext. vs Poisson Sharpe : {(f[2]-b[2])/abs(b[2])*100:+.1f}%")
    print(f"  Mean PnL lift (full vs naive): +${f[0]-b[0]:.3f}")
    print(f"  VaR5% lift (full vs naive)   : +${f[4]-b[4]:.3f}")
    print("=" * 68)


if __name__ == "__main__":
    print_summary()
    print("Generating figure ...")
    plot_all()
    print("Done.")

  CJR HAWKES EXECUTION  (directional-run scenario, 600 paths)
  Model                     Mean PnL     Std   Sharpe   InvStd     VaR5%
--------------------------------------------------------------------
  Naive Poisson               25.747   8.382    3.072    3.079    13.258
  CJR Hawkes (Part 1)         26.164   8.413    3.110    3.037    13.291
  Full extension (Part 2)     26.189   7.998    3.275    2.958    13.367
  CJR vs Poisson   Sharpe gain: +1.2%
  Full ext. vs Poisson Sharpe : +6.6%
  Mean PnL lift (full vs naive): +$0.442
  VaR5% lift (full vs naive)   : +$0.109
Generating figure ...
Saved: cartea_jaimungal_ricci_hawkes.png
Done.
